<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI_Phase22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import os, sys, json, ast, hashlib, subprocess, shutil, re, textwrap

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

MYDRIVE = Path(os.environ.get('ICHI2027_MYDRIVE', '/content/drive/MyDrive')).expanduser().resolve()
assert MYDRIVE.is_dir(), 'Mount the original Google Drive containing Phase20 and Phase21 outputs. No Colab browser upload is used.'
PROJECT = MYDRIVE / 'NeuroFHIR_ICHI2027'
OUT = PROJECT / 'phase22_outputs'
PRIVATE = OUT / '_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True, exist_ok=True)
PUBLIC_JSON = OUT / 'ICHI2027_Phase22_SHAREABLE.json'
PRIVATE_JSON = PRIVATE / 'ICHI2027_Phase22_ORIGINAL_PRODUCER_INSPECTION_PRIVATE.json'
PRIVATE_REVIEW = PRIVATE / 'ICHI2027_Phase22_ORIGINAL_PRODUCER_CODE_REVIEW_PRIVATE.py'
PIN_COMMIT = 'a4b7265adabf096e84b957de068e24a5ab853425'
RUNNER_SHA = '5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19'
RUNNER_NAME = 'FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15.ipynb'
TARGET_COLUMNS = {'fidelity_status','warning_code','source_resource_type','source_df_index',
                  'target_omop_table','target_omop_record_id','is_mapping_event'}
ORIGINAL_CONSTANTS = {'LOSS_STATUSES','NON_LINEAGE_ELIGIBLE_STATUSES'}

def utcnow(): return datetime.now(timezone.utc).isoformat()
def file_hash(p):
    h = hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
def content_hash(s): return hashlib.sha256(s.encode('utf-8')).hexdigest()
def run_cmd(args, cwd=None, timeout=240):
    r = subprocess.run(args, cwd=str(cwd) if cwd else None, capture_output=True, text=True, timeout=timeout)
    if r.returncode: raise RuntimeError(f'{args[0]} failed: {r.stderr[-400:]}')
    return r.stdout.strip()
def as_source(raw): return ''.join(raw) if isinstance(raw,list) else (raw or '')

print('Phase22: inspect the EXACT pinned pre-v15 ORIGINAL generic TFL audit producers, recover v15 constants only from original evidence, and validate a faithful producer contract. No new MRI comparator scores.')



Mounted at /content/drive
Phase22: inspect the EXACT pinned pre-v15 ORIGINAL generic TFL audit producers, recover v15 constants only from original evidence, and validate a faithful producer contract. No new MRI comparator scores.


In [2]:
P21_PATH = PROJECT/'phase21_outputs'/'ICHI2027_Phase21_SHAREABLE.json'
M21_PATH = PROJECT/'phase21_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase21_PRIVATE_FROZEN_V15_METHOD_MANIFEST.json'
P20_PATH = PROJECT/'phase20_outputs'/'ICHI2027_Phase20_SHAREABLE.json'
M20_PATH = PROJECT/'phase20_outputs'/'_PRIVATE_DO_NOT_SHARE'/'ICHI2027_Phase20_PRIVATE_C_ADAPTER_MANIFEST.json'
for required in [P21_PATH,M21_PATH,P20_PATH,M20_PATH]:
    assert required.is_file(), f'Keep original same-Drive file available: {required.name}. Do not upload private artifacts to ChatGPT.'
p21=json.loads(P21_PATH.read_text(encoding='utf-8'))
m21=json.loads(M21_PATH.read_text(encoding='utf-8'))
p20=json.loads(P20_PATH.read_text(encoding='utf-8'))
m20=json.loads(M20_PATH.read_text(encoding='utf-8'))
assert p21['phase']=='21_restricted_original_v15_common_domain_real_MRI'
assert m21['phase']==p21['phase'] and p21['run_utc']==m21['run_utc']
assert p21['original_method']['runner_sha256']==RUNNER_SHA
assert m21['verified_original_commit']==PIN_COMMIT
assert m21['verified_original_runner_sha256']==RUNNER_SHA
assert m21['source_phase20_public_sha256']==file_hash(P20_PATH)
assert m21['phase20_private_manifest_sha256']==file_hash(M20_PATH)
assert p20['frozen_original_generic_v15']['repository_commit']==PIN_COMMIT
assert p20['frozen_original_generic_v15']['runner_sha256']==RUNNER_SHA
assert p21['gates']['real_4128_numeric_pairs_per_arm_reconciled'] is True
assert p21['gates']['true_four_arm_comparison_completed'] is False
print('Authentic Phase20→21 public/private provenance: verified. Original v15 identity unchanged.')



Authentic Phase20→21 public/private provenance: verified. Original v15 identity unchanged.


In [3]:
# Reuse the same attested repository when possible; otherwise clone exact frozen COMMIT.
repo_override=os.environ.get('ICHI2027_FROZEN_TFL_REPO')
options=([Path(repo_override)] if repo_override else []) + [
    Path('/content/ichi2027_phase21_pinned_generic_tfl'),
    Path('/content/ichi2027_phase20_pinned_generic_tfl'),
    Path('/content/ichi2027_phase22_pinned_generic_tfl')]
REPO=None
for candidate in options:
    if (candidate/'.git').exists():
        try:
            if run_cmd(['git','rev-parse','HEAD'],candidate,20)==PIN_COMMIT and file_hash(candidate/RUNNER_NAME)==RUNNER_SHA:
                REPO=candidate;break
        except Exception: pass
if REPO is None:
    REPO=Path('/content/ichi2027_phase22_pinned_generic_tfl')
    if REPO.exists(): shutil.rmtree(REPO)
    run_cmd(['git','clone','--quiet','https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git',str(REPO)],timeout=300)
    run_cmd(['git','checkout','--quiet','--detach',PIN_COMMIT],REPO,90)
assert run_cmd(['git','rev-parse','HEAD'],REPO,30)==PIN_COMMIT
RUNNER=REPO/RUNNER_NAME
assert RUNNER.is_file() and file_hash(RUNNER)==RUNNER_SHA, 'Exact original v15 checksum changed: STOP.'
original_files=[REPO/Path(x) for x in run_cmd(['git','ls-files'],REPO,30).splitlines()]
CODE_FILES=[p for p in original_files if p.suffix.lower() in {'.ipynb','.py'} and p.is_file() and p.stat().st_size<=6_000_000]
assert CODE_FILES, 'No source-code files at pinned commit; cannot substitute another revision.'
print('Original SHA-attested v15 runner and pinned git tree verified; source files for STATIC archaeology:',len(CODE_FILES))



Original SHA-attested v15 runner and pinned git tree verified; source files for STATIC archaeology: 25


In [4]:
# AST-only code review. Never execute historical notebooks or reuse their saved execution outputs.
def literal_tokens(node):
    try:
        x=ast.literal_eval(node)
        if isinstance(x,(set,frozenset,list,tuple)) and all(isinstance(z,str) for z in x):return sorted(set(x))
    except Exception: pass
    if (isinstance(node,ast.Call) and isinstance(node.func,ast.Name)
        and node.func.id in {'set','frozenset','tuple','list'} and len(node.args)==1 and not node.keywords):
        return literal_tokens(node.args[0])
    return None

def inspect_fragment(source, relative_name, cell_index):
    try: tree=ast.parse(source)
    except SyntaxError:
        return {'file':relative_name,'cell_index':cell_index,'parse_ok':False,'assigned_columns':[],
                'dict_columns':[],'read_methods':[],'constants':{},'function_names':[],
                'status_literals':[],'evidence':'AST_PARSE_ERROR','source_sha256':content_hash(source)}
    assigned=set(); dict_cols=set(); read_methods=set(); constants={}; functions=[]; status_tokens=set()
    for node in ast.walk(tree):
        if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)): functions.append(node.name)
        if isinstance(node,(ast.Assign,ast.AnnAssign)):
            targets=(node.targets if isinstance(node,ast.Assign) else [node.target]);v=node.value
            for t in targets:
                if isinstance(t,ast.Name) and t.id in ORIGINAL_CONSTANTS:
                    constants[t.id]=literal_tokens(v)
                if isinstance(t,ast.Subscript) and isinstance(t.slice,ast.Constant) and t.slice.value in TARGET_COLUMNS:
                    assigned.add(t.slice.value)
        if isinstance(node,ast.Dict):
            for k in node.keys:
                if isinstance(k,ast.Constant) and isinstance(k.value,str) and k.value in TARGET_COLUMNS:
                    dict_cols.add(k.value)
        if isinstance(node,ast.Call) and isinstance(node.func,ast.Attribute):
            if node.func.attr in {'read_parquet','read_csv','to_parquet','to_csv','assign','merge','map','apply'}:
                read_methods.add(node.func.attr)
            if node.func.attr=='assign':
                for kw in node.keywords:
                    if kw.arg in TARGET_COLUMNS:assigned.add(kw.arg)
        if isinstance(node,ast.Constant) and isinstance(node.value,str):
            tok=node.value.strip()
            if tok.isupper() and re.fullmatch('[A-Z][A-Z0-9_]{2,70}',tok) and (
                    any(s in tok for s in ('LOSS','PRESERV','MAPP','NORMAL','DROP','AMBIG','TRACE'))):
                status_tokens.add(tok)
    defined_status=('fidelity_status' in assigned or 'fidelity_status' in dict_cols)
    defined_warning=('warning_code' in assigned or 'warning_code' in dict_cols)
    defined_target=bool({'target_omop_table','target_omop_record_id'} & (assigned|dict_cols))
    kind=('STATUS_AND_WARNING_OUTPUT_CANDIDATE' if defined_status and defined_warning
          else 'STATUS_OUTPUT_CANDIDATE' if defined_status else 'NO_STATUS_OUTPUT_EVIDENCE')
    return {'file':relative_name,'cell_index':cell_index,'parse_ok':True,
            'assigned_columns':sorted(assigned),'dict_columns':sorted(dict_cols),'read_methods':sorted(read_methods),
            'constants':constants,'function_names':sorted(set(functions)),
            'status_literals':sorted(status_tokens),'evidence':kind,'target_output_signal':defined_target,
            'source_sha256':content_hash(source)}

FRAGMENTS=[]; FRAGMENT_SOURCES={}; errors=[]
for p in CODE_FILES:
    rel=p.relative_to(REPO).as_posix()
    try:
        if p.suffix.lower()=='.ipynb':
            nb=json.loads(p.read_text(encoding='utf-8'))
            parts=[(i,as_source(cell.get('source'))) for i,cell in enumerate(nb.get('cells',[])) if cell.get('cell_type')=='code']
        else: parts=[(None,p.read_text(encoding='utf-8'))]
    except Exception as e:
        errors.append({'file':rel,'error_type':type(e).__name__});continue
    for i,src in parts:
        f=inspect_fragment(src,rel,i)
        FRAGMENTS.append(f)
        FRAGMENT_SOURCES[(rel,i)]=src
assert FRAGMENTS
print('Parsed original frozen-tree code fragments:',len(FRAGMENTS),'; unparseable:',sum(not x['parse_ok'] for x in FRAGMENTS))



Parsed original frozen-tree code fragments: 233 ; unparseable: 2


In [5]:
# Identify ORIGINAL event producer candidates. A reference to "fidelity_status" is NOT evidence of producing an event.
producers=[f for f in FRAGMENTS if f['evidence']!='NO_STATUS_OUTPUT_EVIDENCE' and f['parse_ok']]
producers.sort(key=lambda f: (
    0 if f['evidence']=='STATUS_AND_WARNING_OUTPUT_CANDIDATE' else 1,
    0 if f['target_output_signal'] else 1,
    0 if re.search(r'TFL|fidelity',f['file'],re.I) else 1,
    f['file'],f['cell_index'] if f['cell_index'] is not None else -1))
# Private extracts are original PUBLIC repository code, not generated alternative algorithms.
review=['# Private review: EXACT public pinned repository code fragments, unchanged.',' # Static only; do NOT execute this file; no MRI result comes from old saved notebook outputs.',
        '# Frozen commit '+PIN_COMMIT,'# Frozen v15 runner '+RUNNER_SHA,'']
for index,f in enumerate(producers[:35]):
    src=FRAGMENT_SOURCES[(f['file'],f['cell_index'])]
    review.extend(['\n# === ORIGINAL FRAGMENT '+str(index+1)+' ===',
                   '# file: '+f['file']+'; source code cell index: '+str(f['cell_index']),
                   '# source sha256: '+f['source_sha256'],src,''])
PRIVATE_REVIEW.write_text('\n'.join(review),encoding='utf-8')
# Enumerate exact ORIGINAL class constants; cross-file aliases or conflicts MUST NOT be silently resolved.
constant_origins=defaultdict(list)
for f in FRAGMENTS:
    for k,v in f['constants'].items():
        constant_origins[k].append({'file':f['file'],'cell_index':f['cell_index'],'literal_tokens':v})
producer_public=[{'source_file':f['file'],'code_cell_index_zero_based':f['cell_index'],
                  'source_sha256':f['source_sha256'],'static_evidence':f['evidence'],
                  'target_columns_assigned':f['assigned_columns'], 'record_dictionary_fields':f['dict_columns'],
                  'has_target_output_signal':f['target_output_signal']} for f in producers[:25]]
print('Original source fragments with STATUS assignment evidence:',len(producers),
      '; original exact producer fragments requiring human method review:',len(producer_public))



Original source fragments with STATUS assignment evidence: 6 ; original exact producer fragments requiring human method review: 6


In [6]:
# Verify the *last* original v15 metric function from SOURCE, including Phase21's previously saved fingerprint.
runner_nb=json.loads(RUNNER.read_text(encoding='utf-8'))
RUNNER_CODE=[(i,as_source(c.get('source'))) for i,c in enumerate(runner_nb['cells']) if c['cell_type']=='code']
assert len(RUNNER_CODE)==17
original_funcs={}; runner_constants={}
for i,src in RUNNER_CODE:
    tree=ast.parse(src)
    for node in tree.body:
        if isinstance(node,ast.FunctionDef) and node.name in {'compute_metrics','nonempty_warning','split_warning_tokens'}:
            original_funcs[node.name]={'cell_index':i,'source':ast.get_source_segment(src,node)}
        if isinstance(node,(ast.Assign,ast.AnnAssign)):
            targets=node.targets if isinstance(node,ast.Assign) else [node.target]
            for t in targets:
                if isinstance(t,ast.Name) and t.id in ORIGINAL_CONSTANTS:
                    runner_constants.setdefault(t.id,[]).append({'cell_index':i,'tokens':literal_tokens(node.value)})
assert original_funcs['compute_metrics']['cell_index']==p21['original_method']['final_original_compute_metrics_cell_index']
metric_source_hash=content_hash(original_funcs['compute_metrics']['source'])
assert metric_source_hash==p21['original_method']['last_original_metric_source_sha256']
exact_runner_constants={}
for name in ORIGINAL_CONSTANTS:
    usable=[entry['tokens'] for entry in runner_constants.get(name,[]) if entry['tokens'] is not None]
    # Only ORIGINAL v15 runner constants count here; older notebook names are not equivalent unless formally attested.
    if usable and all(v==usable[-1] for v in usable):exact_runner_constants[name]=usable[-1]
if 'LOSS_STATUSES' in exact_runner_constants and 'NON_LINEAGE_ELIGIBLE_STATUSES' in exact_runner_constants:
    assert 'UNMAPPED' in exact_runner_constants['NON_LINEAGE_ELIGIBLE_STATUSES']
status_evidence=sorted({tok for f in producers if 'fidelity_status' in (set(f['assigned_columns'])|set(f['dict_columns']))
                        for tok in f['status_literals']})
proven_safe_status=[tok for tok in status_evidence if tok in {'PRESERVED','UNCHANGED','EXACT','NORMALIZED'}
                    and tok not in exact_runner_constants.get('LOSS_STATUSES',[])
                    and tok not in exact_runner_constants.get('NON_LINEAGE_ELIGIBLE_STATUSES',[])]
print('Last frozen v15 function SHA reverified:',metric_source_hash,
      '; exact final-runner class constants recovered:', sorted(exact_runner_constants),
      '; static original producer candidate statuses:', proven_safe_status)



Last frozen v15 function SHA reverified: ad75e76182f8a5e2b71616b563a24203a3160e8b8bad28c537ec7279ec2efab2 ; exact final-runner class constants recovered: ['LOSS_STATUSES', 'NON_LINEAGE_ELIGIBLE_STATUSES'] ; static original producer candidate statuses: ['NORMALIZED', 'PRESERVED']


In [7]:
# A strictly separated SYNTHETIC metric-contract microtest is optional. Do not create MRI scores from unavailable class constants.
MICROTEST={'attempted':False,'passed':False,'reason':None,'synthetic_only':True}
ready_constants=all(x in exact_runner_constants for x in ORIGINAL_CONSTANTS)
if ready_constants and proven_safe_status:
    import numpy as np
    import pandas as pd
    originals={'np':np,'pd':pd}
    originals.update({k:set(v) for k,v in exact_runner_constants.items()})
    for nm in ('nonempty_warning','compute_metrics'):
        exec(compile(ast.parse(original_funcs[nm]['source']),f'PINNED_V15_ORIGINAL_{nm}','exec'),originals)
    preserved=proven_safe_status[0]
    fixture=pd.DataFrame([
      dict(source_resource_type='Observation',source_df_index=0,target_omop_table='SYNTHETIC_ONLY',target_omop_record_id=1,is_mapping_event=True,fidelity_status=preserved,warning_code=None),
      dict(source_resource_type='Observation',source_df_index=1,target_omop_table='SYNTHETIC_ONLY',target_omop_record_id=2,is_mapping_event=True,fidelity_status='TRACEABILITY_LOSS',warning_code=None),
      dict(source_resource_type='Observation',source_df_index=2,target_omop_table=None,target_omop_record_id=None,is_mapping_event=False,fidelity_status='UNMAPPED',warning_code=None)
    ])
    got=originals['compute_metrics'](fixture)
    assert got['audit_events']==3 and got['unique_source_resources']==3
    assert got['mapped_source_resources']==2 and got['lineage_eligible_events']==2
    assert got['traceability_loss_events']==1 and got['lineage_coverage']==0.5
    MICROTEST={'attempted':True,'passed':True,'synthetic_only':True,
               'original_unmodified_metric_function_sha256':metric_source_hash,
               'original_fate_status_used':preserved,'synthetic_audit_rows':3}
else:
    why=[]
    if not ready_constants:why.append('The pinned v15 runner does not contain an unambiguous literal definition of both original fate constants.')
    if not proven_safe_status:why.append('No original producer-derived mapped/preserved status was authoritatively identified.')
    MICROTEST['reason']=' '.join(why)
print('Original metric synthetic microtest performed:',MICROTEST['attempted'],
      '; real MRI generic Arm C executed: FALSE')



Original metric synthetic microtest performed: True ; real MRI generic Arm C executed: FALSE


In [8]:
# Make bridge requirements explicit. Source observations are real; no invented original audit rows or mappings.
BRIDGE={
 'numeric_source_identity':{'evidence':'PHASE21_REAL_FHIR_TO_PRIVATE_CANDIDATE_RECONCILED','original_generic_status_assigned':False},
 'original_generic_source_to_target_event_generator':{'original_static_candidates':len(producers),
       'executed_on_current_MRI':False,'requires_manual_verified_producer_contract':True},
 'original_v15_aggregate_metric_function':{'source_verified':True,'sha256':metric_source_hash,
       'original_fate_class_constants_approved':ready_constants},
 'original_frozen_generic_C_imaging_provenance':{'not_automatically_supported_by_general_medication_audit':True},
 'regenerated_TRACE_subject_pair_state':{'source_observations':1806,'generic_original_rule_verified':False},
 'unverified_original_export_units':{'do_not_score_as_ETL_loss':True},
 'unverified_DICOM_identifiers':{'do_not_claim_official_MI_CDM_load':True},
 'athena_approved_standard_concepts':{'snapshot_currently_absent':True}}
GATES={
 'original_v15_commit_and_runner_sha256_verified':True,
 'Phase20_Phase21_source_and_private_provenance_verified':True,
 'original_pre_v15_code_tree_statically_inspected':True,
 'status_output_source_fragments_located':bool(producers),
 'one_exact_original_generic_audit_producer_semantics_human_approved':False,
 'all_required_original_v15_constant_definitions_unambiguously_recovered':ready_constants,
 'unchanged_v15_original_metric_synthetic_microtest_passed':MICROTEST['passed'],
 'current_real_MRI_genuine_generic_C_event_generator_executed':False,
 'current_real_MRI_generic_full_arm_C_completed':False,
 'exact_ADNI_original_raw_source_units_approved':False,
 'original_DICOM_UID_semantics_approved':False,
 'Athena_vocabulary_concept_mappings_approved':False,
 'official_MI_CDM_load':False,
 'valid_full_four_arm_inference':False}
# A source fragment containing a dict/assignment is a candidate, never an automatically approved generic producer.
assert not GATES['one_exact_original_generic_audit_producer_semantics_human_approved']
print('Original generic source producer approval: PENDING. Four-arm results: NOT AVAILABLE.')



Original generic source producer approval: PENDING. Four-arm results: NOT AVAILABLE.


In [9]:
PUBLIC={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'22_pinned_original_generic_TFL_producer_and_constants_archaeology',
 'run_utc':utcnow(),
 'status':'PINNED_ORIGINAL_SOURCE_PRODUCER_STATIC_REVIEW_AND_OPTIONAL_SYNTHETIC_METRIC_MICROTEST_NO_FULL_MRI_ARM_C',
 'privacy':'Aggregate and public repository source locations only. Original public code excerpts and all private research source links remain in private Drive.',
 'phase21_upstream_verified':True,
 'original_repository':{'verified_commit':PIN_COMMIT,'verified_frozen_v15_runner_sha256':RUNNER_SHA,
                        'original_code_files_inspected':len(CODE_FILES),'original_code_fragments_inspected':len(FRAGMENTS),
                        'unparseable_fragments':sum(not f['parse_ok'] for f in FRAGMENTS)},
 'original_generic_event_producer_recovery':{
   'static_status_output_candidate_count':len(producers),
   'candidate_public_source_locations':producer_public[:15],
   'a_complete_executable_original_generic_MRI_generator_recovered':False,
   'original_source_code_automatically_executed_on_MRI':False,
   'interpretation':'AST output assignments mark possible source locations; they do not establish that old medication-aware methods cover MRI or TRACE.'},
 'original_v15_method_attestation':{
   'last_original_metric_cell_index':original_funcs['compute_metrics']['cell_index'],
   'last_original_metric_function_sha256':metric_source_hash,
   'runner_defined_class_constant_names':sorted(exact_runner_constants),
   'literal_constant_candidate_counts_from_other_pinned_sources':{k:len(v) for k,v in sorted(constant_origins.items())},
   'earlier_source_values_NOT_automatically_substituted_into_v15':True,
   'original_producer_candidate_nonloss_statuses_for_manual_review':proven_safe_status[:8]},
 'synthetic_original_unchanged_metric_microtest':MICROTEST,
 'MRI_source_domain_inherited_from_phase21':{'numeric_events_per_arm':4128,
    'original_regenerated_TRACE_pair_feature_assessments':1806,
    'genuine_source_to_official_MI_CDM_mapping_implemented':False},
 'frozen_generic_C_real_MRI_results':None,
 'official_MI_CDM_load_results':None,
 'real_four_arm_comparator_scores':None,
 'gates':GATES,
 'next_stage_gate':('REVIEW_EXACT_ORIGINAL_SOURCE_PRODUCER_CANDIDATE_CODE_AND_FREEZE_A_SEMANTICALLY_FAITHFUL_MRI_INPUT_ADAPTER; '
                  'DO_NOT_RUN_A_NEWLY_INVENTED_GENERIC_RULESET_AS_ORIGINAL_V15; '
                  'OBTAIN_ORIGINAL_ADNI_EXPORT_AND_ATHENA_BEFORE_OFFICIAL_MI_CDM')}
PRIVATE_MANIFEST={
 'public_report':PUBLIC,'phase21_public_file_sha256':file_hash(P21_PATH),
 'phase21_private_manifest_file_sha256':file_hash(M21_PATH),
 'phase20_public_file_sha256':file_hash(P20_PATH),
 'pinned_runner_file_sha256':file_hash(RUNNER),
 'full_static_status_output_producer_candidates':producers,
 'literal_original_constants_and_locations':dict(constant_origins),
 'runner_literal_fate_constants_and_locations':runner_constants,
 'unparseable_or_unreadable_files':errors,
 'bridge_status':BRIDGE,
 'original_source_fragment_code_review_private':str(PRIVATE_REVIEW),
 'original_source_fragment_code_review_sha256':file_hash(PRIVATE_REVIEW)}
PRIVATE_JSON.write_text(json.dumps(PRIVATE_MANIFEST,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
PUBLIC_JSON.write_text(json.dumps(PUBLIC,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
assert PUBLIC_JSON.is_file() and PRIVATE_JSON.is_file() and PRIVATE_REVIEW.is_file()
assert not any(leak in PUBLIC_JSON.read_text(encoding='utf-8') for leak in ('/content/drive','_PRIVATE_DO_NOT_SHARE','urn:uuid:','SOURCE_RID'))
print('SHAREABLE:',PUBLIC_JSON)
print('PRIVATE method inspection and unchanged public-source-code review: your Google Drive only.')
print('Send only ICHI2027_Phase22_SHAREABLE.json here; the private producer review copy may be shared separately AFTER inspection.')


SHAREABLE: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase22_outputs/ICHI2027_Phase22_SHAREABLE.json
PRIVATE method inspection and unchanged public-source-code review: your Google Drive only.
Send only ICHI2027_Phase22_SHAREABLE.json here; the private producer review copy may be shared separately AFTER inspection.
